# §2.3 Proof of Concept — Figure Candidates (Roth-Erev focus, v3)

This notebook is a Roth-Erev-focused subset of
[`proof_of_concept_figures_v2.ipynb`](proof_of_concept_figures_v2.ipynb).
It drops Option D-γ (2D heatmap) entirely, refocuses Option F on Roth-Erev
alone (Q-learning's flat-curve story is deferred to §4), and adds a
combined view that places Option D-β and the new Option F side by side.

Runs **locally** or on **Google Colab**, controlled by the
`RUNNING_LOCALLY` switch in the first code cell:

- **Local** (`RUNNING_LOCALLY = True`): figures are displayed inline
  *and* saved as PNGs under `../results/proof_of_concept/`.
- **Colab** (`RUNNING_LOCALLY = False`): the bootstrap cells clone the
  repo, `pip install -e .` it, **mount Google Drive**, and save PNGs +
  CSVs to a project folder there. Use Colab when you want to crank up
  `N_SEEDS_OPT_A` / `BASIN_N_SEEDS` / `N_EPISODES` past what your laptop
  can comfortably run.

## The v3 shortlist at a glance

| # | Name | What it shows |
|---|---|---|
| 1 | Initialization sweep (rewards + NMI) | Time-series per init regime; the basin-reachability story. |
| A | Phase-portrait trajectories | Same runs as Fig. 1 but as motion in (NMI, reward) space. |
| D-β | Basin of attraction (mean ± std curves) | Continuous `sig_n` sweep at H=10,000, reward and NMI overlaid. |
| E | Roth–Erev vs Q-learning side-by-side | D-β-style plot for both agents on shared axes. |
| F | Time-horizon sweep (Roth-Erev) | Reward and NMI vs `sig_n` with one curve per horizon and std bands. |
| F + D-β combined | Side-by-side composite | Deep asymptotic (D-β) next to the horizon ladder (F). |

Set `SMOKE_TEST = True` in the parameters cell for fast iteration; note
that Option F's max horizon is 10,000 episodes, so `N_EPISODES` must be
at least that — `SMOKE_TEST` clips `N_EPISODES` to 3,000 and will trip
Option F's assertion. Run Option F at the default `N_EPISODES` only.


## Environment setup

Three small cells before anything else:

1. **Environment switch** — `RUNNING_LOCALLY` decides everything that
   follows. On local: notebook's parent is the repo root and is added
   to `sys.path`; `RESULTS_DIR` points at `../results/proof_of_concept/`
   so PNGs are saved there. On Colab: Drive is mounted and `RESULTS_DIR`
   points at a project folder under `My Drive`, so PNGs and CSVs persist
   across runtimes. The next two cells handle the clone + install.
2. **Git clone + chdir** — only fires on Colab. Force-fresh clone, then
   `os.chdir` into the clone and put it on `sys.path`. Uses Python
   builtins (`os.chdir`, `subprocess.run`) rather than line magics
   (`%cd`, `!pip`) so the `if not RUNNING_LOCALLY:` guard actually
   works (line magics fire regardless of the surrounding `if`).
3. **Pip install** — only fires on Colab. `pip install -q -e .` so the
   `rl_signaling` package and the `analytics.scripts.*` namespace
   become importable.

If you want to run on Colab, update `REPO_URL` in the clone cell to
match wherever this repo lives publicly.


In [ ]:
"""Environment switch — local vs Colab."""

import os
import sys
from pathlib import Path

# True  → laptop run; PNGs save to ../results/proof_of_concept/
# False → Google Colab; PNGs are NOT saved, only displayed inline
RUNNING_LOCALLY = False

if RUNNING_LOCALLY:
    # Notebook lives in <repo>/notebooks/; repo root is the parent.
    REPO_ROOT = Path(os.getcwd()).resolve().parent
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    RESULTS_DIR = REPO_ROOT / "results" / "proof_of_concept"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Local mode.")
    print(f"  REPO_ROOT   = {REPO_ROOT}")
    print(f"  RESULTS_DIR = {RESULTS_DIR}  (PNGs and CSVs will be saved here)")
else:
    # Colab: mount Drive and save artifacts to a project folder under My Drive.
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = Path(
        "/content/drive/My Drive/Colab Projects/Python ABMs/"
        "Distributed Signaling/Plots and Datasets/Proof of Concept/"
    )
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Colab mode.")
    print(f"  RESULTS_DIR = {RESULTS_DIR}  (PNGs and CSVs will be saved to Drive)")

print(f"  CPU cores   = {os.cpu_count()}")


In [ ]:
"""Git clone + chdir + sys.path — Colab only."""

REPO_URL = "https://github.com/IgnacioOQ/RL_Signaling"
REPO_BRANCH = "debugging"   # <-- change when this work merges to main
REPO_NAME = "RL_Signaling"

if not RUNNING_LOCALLY:
    import shutil
    import subprocess

    if os.path.exists(REPO_NAME):
        shutil.rmtree(REPO_NAME)
    subprocess.run(
        ["git", "clone", "-b", REPO_BRANCH, REPO_URL],
        check=True,
    )
    os.chdir(REPO_NAME)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    print(f"Cloned {REPO_URL} (branch: {REPO_BRANCH})")
    print(f"  cwd = {os.getcwd()}")


In [ ]:
"""Pip install — Colab only."""

if not RUNNING_LOCALLY:
    import subprocess
    subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
    print("Installed rl_signaling (editable).")


## Parameters

Every simulation knob lives in the cell below.

**Initialization regimes.** Each `InitSpec` carries independent `(n, m)`
weights for the **signaling urn** and the **action urn**. Across all
four regimes here, the action urn is always initialized uniformly to
`(1, 1)` — only the signaling urn varies. So every regime asks the same
question — "starting from a uniform action policy, how reliably does
learning find a high-reward joint policy?" — under different amounts
of initial signaling pre-coordination, from one-hot deterministic
(`sig=[1,0]`) to fully unbiased (`sig=[1,1]`) to strongly pre-biased
(`sig=[100,1]`). Labels show signaling weights only.

If you're running on Colab, this is where you'd bump up `N_SEEDS_FIG2`
or `N_SEEDS_OPT_A` to take advantage of the extra cores.


In [ ]:
"""Notebook-level parameters."""

from collections import namedtuple

# Flip to True for fast iteration: smaller seed counts, fewer episodes.
SMOKE_TEST = False

# Time horizon per run. (10k is enough for the trajectories to stabilize;
# 30k was overkill on a laptop.)
N_EPISODES = 10_000 if not SMOKE_TEST else 3_000

# Per-figure seed counts.
N_SEEDS_FIG1   = 1                                # one trajectory per init
N_SEEDS_FIG2   = 200 if not SMOKE_TEST else 20    # per-seed scatter
N_SEEDS_OPT_A  = 15  if not SMOKE_TEST else 3     # phase-portrait (excludes sig=[1,0])
N_SEEDS_OPT_B  = 6   if not SMOKE_TEST else 3     # per-cell concentration
GAME_SEED_OPT_C = 0                               # enumeration is deterministic

# Basin sweep (Option D-α, D-β) — continuous sig_n sweep with m_sig=1 and act=(1,1) fixed.
BASIN_SIG_N_VALUES = [1, 2, 3, 5, 8, 13, 25, 50, 100] if not SMOKE_TEST else [1, 5, 50]
# More seeds on Colab where the parallelism is essentially free.
BASIN_N_SEEDS = 10 if SMOKE_TEST else (50 if RUNNING_LOCALLY else 200)

# Basin grid (Option D-γ) — 2D heatmap over (sig_n, act_n). Relaxes act=(1,1).
if SMOKE_TEST:
    GRID_SIG_N_VALUES = [1, 5, 50]
    GRID_ACT_N_VALUES = [1, 5, 50]
    GRID_N_SEEDS = 5
elif RUNNING_LOCALLY:
    GRID_SIG_N_VALUES = [1, 2, 5, 13, 50]
    GRID_ACT_N_VALUES = [1, 2, 5, 13, 50]
    GRID_N_SEEDS = 20
else:  # Colab
    GRID_SIG_N_VALUES = [1, 2, 3, 5, 8, 13, 25, 50, 100]
    GRID_ACT_N_VALUES = [1, 2, 3, 5, 8, 13, 25, 50, 100]
    GRID_N_SEEDS = 50
REWARD_THRESHOLD = 0.9   # cell color = P(final reward > this)

# Q-learning parameters for Option E (the Roth–Erev vs Q-learning comparison).
# Values come from the user's Bayesian-optimization sweep — see
# `notebooks/basic_unit_test.ipynb` for the same agent_kwargs.
QLEARN_PARAMS = {
    "exploration_rate":     0.9652628633727897,
    "exploration_decay":    0.9998122815486062,
    "min_exploration_rate": 1e-10,
    "choice":               "ucb",
    "exp_smoothing":        False,
}

# Smoothing windows for the time-series plots.
WINDOW_REWARD = 100
WINDOW_NMI    = 100

# Parallel workers (-1 = all cores).
N_JOBS = -1

# Initialization regimes. Action urn is always initialized uniformly (1, 1);
# only the signaling urn varies. The label below shows only the signaling
# weights since act is invariant across regimes.
InitSpec = namedtuple("InitSpec", ["label", "sig", "act", "color"])
INITS = [
    InitSpec("sig=[1,0]",   sig=(1, 0),   act=(1, 1), color="tab:blue"),
    InitSpec("sig=[1,1]",   sig=(1, 1),   act=(1, 1), color="tab:orange"),
    InitSpec("sig=[5,1]",   sig=(5, 1),   act=(1, 1), color="tab:green"),
    InitSpec("sig=[100,1]", sig=(100, 1), act=(1, 1), color="tab:red"),
]

# Reader-friendly descriptions for the four initialization regimes.
# Used in figure legends and panel titles (paired with the spec.label
# in parens for traceability, e.g. "Frozen signaling (sig=[1,0])").
INIT_DESC = {
    "sig=[1,0]":   "Frozen signaling",
    "sig=[1,1]":   "Uniform start",
    "sig=[5,1]":   "Mild pre-bias",
    "sig=[100,1]": "Strong pre-bias",
}

print(f"SMOKE_TEST  = {SMOKE_TEST}")
print(f"N_EPISODES  = {N_EPISODES:,}")
print(f"INITS:")
for s in INITS:
    print(f"  {s.label:<14}  sig={s.sig}  act={s.act}")


## Setup — imports, env builder, save helper

This cell imports the canonical `rl_signaling` API plus the one helper
from the analytics scripts (`enumerate_absorbing_rewards` for Option C),
defines the asymmetric-init env builder, and defines a tiny
`save_and_show(filename)` helper that saves PNGs to `RESULTS_DIR` on
local runs and skips the save on Colab.

The other compute helpers (`build_env`, `run_for_A`, `run_for_B`,
`run_one`) under `analytics/scripts/` were written before this notebook
needed asymmetric initialization; using them here would require signature
changes, so we keep them out of the import list.


In [ ]:
"""Imports, the asymmetric-init env builder, and save_and_show."""

import random
from collections import Counter
from contextlib import contextmanager

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from rl_signaling import MultiAgentEnv, UrnAgent, run_simulation
from rl_signaling.agents import QLearningAgent
from rl_signaling.games import create_random_canonical_game, create_initial_signals
from analytics.scripts.figure_poc_options import enumerate_absorbing_rewards

# Canonical §2.3 game shape.
N_FEATURES = 2
N_SIG = 2
N_ACT = 4

%matplotlib inline
plt.rcParams["figure.dpi"] = 110


@contextmanager
def tqdm_joblib(tqdm_obj):
    """Patch joblib's batch-completion callback to drive a tqdm progress bar.

    Usage:
        with tqdm_joblib(tqdm(desc="...", total=len(tasks))):
            results = Parallel(n_jobs=N_JOBS)(delayed(f)(x) for x in tasks)
    """
    class _Cb(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kw):
            tqdm_obj.update(n=self.batch_size)
            return super().__call__(*args, **kw)

    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = _Cb
    try:
        yield tqdm_obj
    finally:
        joblib.parallel.BatchCompletionCallBack = old
        tqdm_obj.close()


def build_env_from_spec(spec, seed: int,
                        agent_type=UrnAgent, extra_kwargs=None) -> MultiAgentEnv:
    """Build the canonical 2-agent signaling env with independent (n, m) weights
    for the signaling urn (`spec.sig`) and the action urn (`spec.act`).

    `agent_type` defaults to UrnAgent (Roth–Erev); pass QLearningAgent for the
    Q-learning sweep. `extra_kwargs` are merged into `agent_kwargs` — used to
    pass Q-learning-specific parameters (exploration_rate, choice, etc.)."""
    np.random.seed(seed)
    random.seed(seed)

    graph = nx.DiGraph()
    graph.add_nodes_from([0, 1])
    graph.add_edges_from([(0, 1), (1, 0)])
    games = {i: create_random_canonical_game(N_FEATURES, N_ACT) for i in range(2)}

    agent_kwargs = {"initialize": True, "initialization_weights": spec.sig}
    if extra_kwargs:
        agent_kwargs.update(extra_kwargs)

    env = MultiAgentEnv(
        2, N_FEATURES, N_SIG, N_ACT,
        full_information=False, game_dicts=games,
        observed_variables={0: [0], 1: [1]},
        agent_type=agent_type, graph=graph,
        agent_kwargs=agent_kwargs,
    )

    # If action weights differ from signaling weights, overwrite the action table.
    # Attribute name differs by agent: UrnAgent -> action_urns; QLearningAgent -> q_table_action.
    # IMPORTANT: build a fresh table inside the loop so each agent gets its own
    # dict + arrays — sharing would silently couple the two agents' learning.
    if spec.act != spec.sig:
        n_act, m_act = spec.act
        for agent in env.agents:
            new_action_table = create_initial_signals(
                n_observed_features=2,   # 1 own feature + 1 received signal
                n_signals=N_ACT,
                n=n_act,
                m=m_act,
            )
            if hasattr(agent, "action_urns"):
                agent.action_urns = new_action_table
            if hasattr(agent, "q_table_action"):
                agent.q_table_action = new_action_table
                # Reset visit counts so UCB doesn't read stale data.
                if hasattr(agent, "action_counts"):
                    agent.action_counts = {
                        state: np.zeros(N_ACT) for state in new_action_table
                    }

    return env


def save_and_show(filename: str, dpi: int = 150) -> None:
    """Save the current figure to RESULTS_DIR/filename and display inline.

    RESULTS_DIR points at the local results folder on laptop runs and at the
    mounted Drive folder on Colab — either way the artifact is saved."""
    if RESULTS_DIR is not None:
        path = RESULTS_DIR / filename
        plt.savefig(path, dpi=dpi)
        print(f"Saved {path}")
    plt.show()


def save_csv(df: pd.DataFrame, filename: str) -> None:
    """Save a DataFrame to RESULTS_DIR/filename (laptop or Drive, depending on mode)."""
    if RESULTS_DIR is not None:
        path = RESULTS_DIR / filename
        df.to_csv(path, index=False)
        print(f"Saved {path}")


print("Setup complete.")


## Figure 1 — Initialization sweep (rewards + NMI)

Four regimes — one Roth–Erev run per regime, `N_EPISODES` episodes each.
Two panels: smoothed reward, smoothed NMI. **In every regime the action
urn is initialized uniformly to `(1, 1)`**; only the signaling urn
varies, and the labels show signaling weights only.

### What the four regimes mean

`init_weights = (n, m)` controls the per-cell pre-seeding of an urn: a
randomly chosen "hot" cell starts with weight `n`, every other cell
starts with weight `m`. Under Roth–Erev's positive-only update
$u[a] \leftarrow \max(0, u[a] + r)$, a cell starting at weight 0 can
**never** grow — so any urn initialized with `m = 0` is one-hot
*forever* (the cell pattern is frozen, even if the magnitudes drift).
This is the lever the four signaling regimes pull on:

- **`sig=[1,0]`** (blue) — signaling urns one-hot bijections from
  `t = 0`. Each agent's signal is a deterministic function of its
  observation forever (NMI = 1.0 from the outset). Action urns start
  uniform; the agent has to *learn* what each `(own_obs, received_signal)`
  key should map to.
- **`sig=[1,1]`** (orange) — signaling urns uniform; learning does all
  the work, both for signaling and for actions.
- **`sig=[5,1]`** (green) — signaling urns mildly pre-biased toward an
  arbitrary bijection (5 vs 1 on the hot cell). Actions still uniform.
- **`sig=[100,1]`** (red) — signaling urns strongly pre-biased; actions
  still uniform.

### Why this design is interesting

Every regime asks the *same* question — "starting from a uniform action
policy, how reliably does the joint chain reach high reward?" — under
different amounts of initial signaling pre-coordination. The varying
factor is the signaling channel's head start; the action channel always
starts from scratch.

- The blue (frozen-signaling) regime is the *upper bound*: signals are
  already a perfect deterministic language; the only thing to learn is
  the action mapping. Conditional on a fixed signal, each
  `(own_obs, received_signal)` key's action urn is a single Pólya urn
  with one correct action (reward 1) and three wrong (reward 0); it
  concentrates on the correct action over time.
- Red and green are *intermediate* cases: signaling can still adapt,
  but starts close to a bijection. Whether this *helps* (faster
  convergence) or *hurts* (locking into a bad bijection that the action
  channel then has to compensate for) is the empirical question.
- Orange is the *minimum coordination* case — pure from-scratch
  learning, both signals and actions starting uniform.

### What to look for

- The blue trajectory should rise rapidly to near 1.0 — easiest
  learning problem (only actions update).
- Blue NMI is pinned at 1.0 throughout.
- Green and orange may dissociate: green can end with *higher NMI* but
  *lower reward* than orange (lock-in to a random bijection vs
  co-adaptation to a useful one). See the
  [paper-draft note](../analytics/docs/Proof%20of%20Concept%20(Paper%20Draft).md).


In [ ]:
%%time
# One trajectory per init.
fig1_histories = {}
for spec in INITS:
    env = build_env_from_spec(spec, seed=0)
    _, rewards, nmi, _, _ = run_simulation(env, N_EPISODES, with_signals=True, plot=False)
    fig1_histories[spec.label] = (spec, rewards[0], nmi[0])

# Rewards panel.
fig, ax = plt.subplots(figsize=(7, 4.5))
for label, (spec, r, _) in fig1_histories.items():
    smoothed = pd.Series(r).rolling(WINDOW_REWARD, min_periods=1).mean()
    ax.plot(smoothed, color=spec.color,
            label=f"{INIT_DESC[label]} ({label})", lw=1.2)
ax.set_xlabel("Episode")
ax.set_ylabel("Average reward per episode (smoothed)")
ax.set_title("Average reward over time, by initial signaling bias")
ax.set_ylim(0, 1.05); ax.legend(loc="lower right")
plt.tight_layout()
save_and_show("initializations_urn_rewards.png")

# NMI panel.
fig, ax = plt.subplots(figsize=(7, 4.5))
for label, (spec, _, mi) in fig1_histories.items():
    smoothed = pd.Series(mi).rolling(WINDOW_NMI, min_periods=1).mean()
    ax.plot(smoothed, color=spec.color,
            label=f"{INIT_DESC[label]} ({label})", lw=1.2)
ax.set_xlabel("Episode")
ax.set_ylabel("NMI per episode (smoothed)")
ax.set_title("Signal informativeness (NMI) over time, by initial signaling bias")
ax.set_ylim(0, 1.05); ax.legend(loc="lower right")
plt.tight_layout()
save_and_show("initializations_urn_nmi.png")


## Option A — Phase-portrait trajectories in (NMI, reward)

**Three panels**, one per non-frozen init regime — `sig=[1,0]` is
dropped because its trajectories sit motionless on the right edge
(NMI = 1.0) and the phase portrait degenerates to vertical motion in
reward. The interesting contrast is among `sig=[1,1]`, `sig=[5,1]`, and
`sig=[100,1]`, which is where the dynamics is doing visible work.

15 seeds per init. Each seed gives an `N_EPISODES`-episode trajectory;
we smooth both reward and NMI with a 500-episode rolling mean, then
plot each trajectory as a series of small dots in the (NMI, reward)
plane, **colored by episode** (viridis: purple = early, yellow = late).
The endpoint is marked with a black `X`.

### What the picture is doing

The same data as Figure 1, but reorganized: instead of "reward over time"
+ "NMI over time" as two parallel lines, we plot the pair
$(\text{NMI}_t, \text{reward}_t)$ as a point that *moves* through the
plane over time. Early-time positions are purple, late-time positions
yellow, and the `X` is where the chain ends up.

This is sometimes called a **phase portrait** — borrowing the term from
dynamical systems, where it shows trajectories of a state moving through
state space.

### What to look for

- `sig=[1,1]` orange: trajectories sweep across the (NMI, reward)
  plane. Some seeds end at high reward / partial NMI (co-adaptation
  succeeded); some end on the left edge near (NMI ≈ 0, reward ≈ 0.5)
  (the no-signaling failure mode revealed by Figure 2).
- `sig=[5,1]` green: trajectories cover less ground than orange and
  end mostly with higher NMI (~0.5–0.95). Endpoint spread on the
  reward axis reflects lock-in to different bijections.
- `sig=[100,1]` red: tight cluster on the right — trajectories barely
  move (start close to where they end up).

### Wrinkle

At the current resolution the trajectories can look like noisy
scribbles because every dot is a 500-episode-smoothed snapshot. With
15 seeds per panel the density is higher than the 8-seed version but
individual lines are still hard to follow through the cluster. Worth
deciding whether the trajectory *texture* (lots of overlapping paths)
is the right visual emphasis, or whether a smaller seed count with
labeled individual trajectories would communicate better.


In [ ]:
%%time
def run_for_A(spec, seed: int) -> dict:
    env = build_env_from_spec(spec, seed)
    _, rewards, nmi, _, _ = run_simulation(env, N_EPISODES, with_signals=True, plot=False)
    r = pd.Series(rewards[0]).rolling(500, min_periods=1).mean().to_numpy()
    n = pd.Series(nmi[0]).rolling(500, min_periods=1).mean().to_numpy()
    return {"spec": spec, "seed": seed, "reward": r, "nmi": n}

OPTA_SPECS = [s for s in INITS if s.label != "sig=[1,0]"]
tasks_A = [(spec, s) for spec in OPTA_SPECS for s in range(N_SEEDS_OPT_A)]
print(f"Running {len(tasks_A)} sims ({N_SEEDS_OPT_A} seeds × {len(OPTA_SPECS)} inits)...")
with tqdm_joblib(tqdm(desc="phase portrait trajectories", total=len(tasks_A))):
    records_A = Parallel(n_jobs=N_JOBS)(
        delayed(run_for_A)(spec, s) for (spec, s) in tasks_A
    )

fig, axes = plt.subplots(1, len(OPTA_SPECS), figsize=(4.3 * len(OPTA_SPECS), 4),
                         sharex=True, sharey=True)
for ax, spec in zip(axes, OPTA_SPECS):
    for rec in [r for r in records_A if r["spec"].label == spec.label]:
        t = np.linspace(0, 1, len(rec["nmi"]))
        ax.scatter(rec["nmi"], rec["reward"], c=t, cmap="viridis", s=1, alpha=0.4)
        ax.scatter(rec["nmi"][-1], rec["reward"][-1], c="black", s=30,
                   marker="X", zorder=10)
    ax.axhline(0.25, ls="--", c="grey", alpha=0.5)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"{INIT_DESC[spec.label]} ({spec.label})")
    ax.set_xlabel("NMI (smoothed)")
axes[0].set_ylabel("Average reward (smoothed)")
fig.suptitle(
    f"Learning trajectories through NMI × reward space  "
    f"({N_SEEDS_OPT_A} trials per panel; color: early (purple) → late (yellow); X = endpoint)",
    fontsize=12,
)
plt.tight_layout()
save_and_show("poc_optionA_phase_portrait.png")


## Option D-β — Basin of attraction (continuous signaling-bias sweep)

This section sweeps the signaling pre-bias parameter `n_sig` continuously
(holding `m_sig = 1` and `act = (1, 1)` fixed) to see how the basin of
attraction of high-reward joint policies depends on the amount of
initial signaling coordination.

**D-β (mean ± std curves)** — x-axis is `sig_n` on a log scale, y-axis
is the final value. Red curve = mean reward across trials, shaded band
= mean ± 1 std (clipped to `[0, 1]`). Green curve / band = same for
NMI. Reward and NMI are overlaid so the dissociation between them is
visible across the sweep — the dropping signaling-pre-bias forces the
chain through the basin boundary.

It answers: "how much signaling pre-bias is needed for the joint chain
to reach high reward reliably?"

The compute cell runs the full sweep once; the plot cell below renders
β from the resulting DataFrame.

**Runtime.** `len(BASIN_SIG_N_VALUES) × BASIN_N_SEEDS` simulations,
parallelized. Default: 9 × 50 = 450 sims, roughly 2 min on a 4-core
laptop. Bump `BASIN_N_SEEDS` (in the Parameters cell) on Colab if you
want tighter bands.


In [ ]:
%%time
"""Run the basin sweep: for each sig_n value, BASIN_N_SEEDS seeds."""

def run_basin_seed(sig_n, sig_m, seed):
    spec = InitSpec(label=f"sig=[{sig_n},{sig_m}]", sig=(sig_n, sig_m),
                    act=(1, 1), color="tab:gray")
    env = build_env_from_spec(spec, seed)
    _, rewards, nmi, _, _ = run_simulation(env, N_EPISODES, with_signals=True, plot=False)
    return {
        "sig_n": sig_n,
        "sig_m": sig_m,
        "seed": seed,
        "final_reward": float(np.mean(rewards[0][-1000:])),
        "final_nmi":    float(np.mean(nmi[0][-1000:])),
    }

tasks_D = [(n, 1, s) for n in BASIN_SIG_N_VALUES for s in range(BASIN_N_SEEDS)]
print(f"Running {len(tasks_D)} sims ({BASIN_N_SEEDS} seeds × {len(BASIN_SIG_N_VALUES)} sig_n values)...")
with tqdm_joblib(tqdm(desc="basin sweep (Roth-Erev)", total=len(tasks_D))):
    records_D = Parallel(n_jobs=N_JOBS)(
        delayed(run_basin_seed)(n, m, s) for (n, m, s) in tasks_D
    )
df_basin = pd.DataFrame(records_D)
save_csv(df_basin, "basin_sweep_data.csv")
print(f"Collected {len(df_basin)} records over sig_n = {BASIN_SIG_N_VALUES}")


In [ ]:
"""Option D-β — per-seed scatter (reward) plus reward and NMI summary curves."""

fig, ax = plt.subplots(figsize=(7.5, 4.8))

# Reward: mean ± std overlay (std band clipped to [0, 1]).
g_r = df_basin.groupby("sig_n")["final_reward"]
mean_r, std_r = g_r.mean(), g_r.std()
ax.plot(mean_r.index, mean_r.values,
        color="firebrick", lw=2, marker="o", label="Reward: mean")
ax.fill_between(mean_r.index,
                np.clip(mean_r.values - std_r.values, 0, 1),
                np.clip(mean_r.values + std_r.values, 0, 1),
                color="firebrick", alpha=0.18, label="Reward: mean ± std")

# NMI: mean ± std overlay (std band clipped to [0, 1]).
g_n = df_basin.groupby("sig_n")["final_nmi"]
mean_n, std_n = g_n.mean(), g_n.std()
ax.plot(mean_n.index, mean_n.values,
        color="darkgreen", lw=2, marker="s", label="NMI: mean")
ax.fill_between(mean_n.index,
                np.clip(mean_n.values - std_n.values, 0, 1),
                np.clip(mean_n.values + std_n.values, 0, 1),
                color="darkgreen", alpha=0.15, label="NMI: mean ± std")

ax.axhline(0.5, ls=":", c="grey", alpha=0.7,
           label="No-signaling reward baseline (≈ 0.5)")
ax.set_xscale("log")
ax.set_xlabel("Initial signaling bias (log scale)")
ax.set_ylabel("Final value (averaged over last 1000 episodes)")
ax.set_title(f"Final reward and NMI vs initial signaling bias  "
             f"({BASIN_N_SEEDS} trials per value)")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right", fontsize=8, ncol=2)
plt.tight_layout()
save_and_show("basin_beta_scatter.png")


## Option E — Roth–Erev vs Q-learning: side-by-side basin comparison

The full Option D analysis runs the basin sweep with `UrnAgent` (Roth–Erev).
This section repeats the **D-β** plot for `QLearningAgent` and shows the two
on shared axes, side-by-side. The question is whether Q-learning's basin is
visibly wider than Roth–Erev's — a likely empirical answer to Reviewer 2's
"why does Q-learning outperform?" question.

### Setup

- **Same `sig_n` grid** as Option D-β: `BASIN_SIG_N_VALUES`.
- **Same `act = (1, 1)` invariant** as Option D-β.
- **Same `BASIN_N_SEEDS` per `sig_n` value.**
- **Q-learning parameters** are taken from the user's earlier Bayesian
  optimization (`QLEARN_PARAMS` in the Parameters cell): UCB choice rule,
  initial exploration ≈ 0.965, decay ≈ 0.9998, floor ≈ 1e-10, no
  exponential smoothing.

### Prediction

Q-learning's exploration bonus (UCB) drives the agent to try untried
signals/actions regardless of where the Q-table started. So even at
`sig_n = 1` (uniform signaling), Q-learning should reach high final
reward, and the spread across seeds should be tight.

In contrast Roth–Erev (left panel) has the lock-in / no-signaling failure
modes we already characterized: low `sig_n` → wide spread, sometimes
reward ≈ 0.5.

If that prediction holds, the side-by-side picture *visually demonstrates*
the §2.3 robustness gap: Roth–Erev's basin reach depends on initial
coordination; Q-learning's doesn't, because exploration substitutes for
coordination.

### Runtime

`len(BASIN_SIG_N_VALUES) × BASIN_N_SEEDS` Q-learning sims — same cost as
Option D-β. The compute cell runs the Q-learning sweep; the plot cell
overlays it next to the existing `df_basin` from Option D-β. **Run
Option D-β's compute cell first**, otherwise `df_basin` will not exist.


In [ ]:
%%time
"""Option E — Q-learning basin sweep. Same sig_n grid and seed count as D-β."""

def run_basin_seed_ql(sig_n, sig_m, seed):
    spec = InitSpec(label=f"sig=[{sig_n},{sig_m}]", sig=(sig_n, sig_m),
                    act=(1, 1), color="tab:gray")
    env = build_env_from_spec(
        spec, seed,
        agent_type=QLearningAgent,
        extra_kwargs=QLEARN_PARAMS,
    )
    _, rewards, nmi, _, _ = run_simulation(env, N_EPISODES, with_signals=True, plot=False)
    return {
        "sig_n": sig_n,
        "sig_m": sig_m,
        "seed": seed,
        "final_reward": float(np.mean(rewards[0][-1000:])),
        "final_nmi":    float(np.mean(nmi[0][-1000:])),
    }

tasks_E = [(n, 1, s) for n in BASIN_SIG_N_VALUES for s in range(BASIN_N_SEEDS)]
print(f"Running {len(tasks_E)} Q-learning sims "
      f"({BASIN_N_SEEDS} seeds × {len(BASIN_SIG_N_VALUES)} sig_n values)...")
with tqdm_joblib(tqdm(desc="basin sweep (Q-learning)", total=len(tasks_E))):
    records_E = Parallel(n_jobs=N_JOBS)(
        delayed(run_basin_seed_ql)(n, m, s) for (n, m, s) in tasks_E
    )
df_basin_ql = pd.DataFrame(records_E)
save_csv(df_basin_ql, "basin_sweep_data_ql.csv")
print(f"Collected {len(df_basin_ql)} Q-learning records over sig_n = {BASIN_SIG_N_VALUES}")


In [ ]:
"""Option E — render the side-by-side D-β-style plot for both agents."""

def render_basin_panel(ax, df, title):
    g_r = df.groupby("sig_n")["final_reward"]
    mean_r, std_r = g_r.mean(), g_r.std()
    ax.plot(mean_r.index, mean_r.values,
            color="firebrick", lw=2, marker="o", label="Reward: mean")
    ax.fill_between(mean_r.index,
                    np.clip(mean_r.values - std_r.values, 0, 1),
                    np.clip(mean_r.values + std_r.values, 0, 1),
                    color="firebrick", alpha=0.18, label="Reward: mean ± std")

    g_n = df.groupby("sig_n")["final_nmi"]
    mean_n, std_n = g_n.mean(), g_n.std()
    ax.plot(mean_n.index, mean_n.values,
            color="darkgreen", lw=2, marker="s", label="NMI: mean")
    ax.fill_between(mean_n.index,
                    np.clip(mean_n.values - std_n.values, 0, 1),
                    np.clip(mean_n.values + std_n.values, 0, 1),
                    color="darkgreen", alpha=0.15, label="NMI: mean ± std")

    ax.axhline(0.5, ls=":", c="grey", alpha=0.7,
               label="No-signaling reward baseline (≈ 0.5)")
    ax.set_xscale("log")
    ax.set_xlabel("Initial signaling bias (log scale)")
    ax.set_title(title)
    ax.set_ylim(0, 1.05)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
render_basin_panel(axes[0], df_basin,    "Roth–Erev")
render_basin_panel(axes[1], df_basin_ql, "Q-learning")
axes[0].set_ylabel("Final value (averaged over last 1000 episodes)")
axes[0].legend(loc="lower right", fontsize=8, ncol=2)
fig.suptitle(f"Final reward and NMI vs initial signaling bias: Roth–Erev vs Q-learning  "
             f"({BASIN_N_SEEDS} trials per value; identical setup except learning rule)",
             fontsize=12)
plt.tight_layout()
save_and_show("basin_e_comparison.png")


## Option F — Time-horizon sweep (Roth-Erev): when does initial bias matter?

Option D-β fixes the horizon at 10,000 episodes — a deep-asymptotic
measurement that hides the dynamics. Option F unfolds the same Roth-Erev
basin sweep across a horizon ladder so the transient story becomes
visible.

For each `sig_n` in `BASIN_SIG_N_VALUES` and each horizon
`H ∈ {10, 50, 100, 300, 1000, 3000, 10000}` we record the final-window
average reward and NMI. The window per horizon is `max(10, min(1000, H // 10))`
— short enough to follow the transient, long enough to suppress single-episode
noise.

### What to look for

- The **reward** curves should rise both with `sig_n` (basin reach) and
  with `H` (chain has time to climb into the basin). Roth-Erev's
  additive updates mean both effects compound; neither washes out.
- The **NMI** curves should be nearly horizon-independent at the
  extremes of `sig_n` (low bias → uniformly low NMI; high bias →
  near-saturation NMI even at short horizons because the initial bias
  is locked in by the urn) and most horizon-sensitive in the
  intermediate region where the chain is still moving.

The 10- and 50-episode curves are the new additions: they sit close to
the initial-policy baseline and answer "how quickly does the chain
*start* climbing toward the basin?" — a question that the original
`H ∈ {100, 300, 1000, 3000, 10000}` set could not answer.

### Runtime

`len(BASIN_SIG_N_VALUES) × BASIN_N_SEEDS` Roth-Erev sims — same cost as
Option D-β (no extra horizons cost anything because every horizon is a
slice of the same 10,000-episode trajectory). Default: 9 × 50 = 450
sims, ~2 min on a 4-core laptop. Q-learning is intentionally dropped
from this section; see Option E for the Roth-Erev vs Q-learning
comparison and §4 (planned) for the Q-learning horizon story.


In [ ]:
%%time
"""Option F (v3) — Roth-Erev only. Sweep both signaling bias and horizon.
Each simulation is run once at N_EPISODES; every horizon in HORIZON_VALUES
is a slice of that single trajectory, so adding horizons is free."""

HORIZON_VALUES = [10, 50, 100, 300, 1000, 3000, 10_000]
# Window per horizon: 10 episodes when horizon is tiny, otherwise H // 10
# capped at 1,000. Matches the Option E / v2 Option F convention.
HORIZON_WINDOWS = {h: max(10, min(1000, h // 10)) for h in HORIZON_VALUES}

assert max(HORIZON_VALUES) <= N_EPISODES, (
    f"Max horizon {max(HORIZON_VALUES)} exceeds N_EPISODES {N_EPISODES}. "
    "Increase N_EPISODES in the Parameters cell or shorten HORIZON_VALUES."
)


def run_horizon_seed(sig_n, sig_m, seed):
    spec = InitSpec(label=f"sig=[{sig_n},{sig_m}]", sig=(sig_n, sig_m),
                    act=(1, 1), color="tab:gray")
    env = build_env_from_spec(spec, seed, agent_type=UrnAgent)
    _, rewards, nmi, _, _ = run_simulation(env, N_EPISODES, with_signals=True, plot=False)
    # Match the agent-0 convention used by the other basin DataFrames.
    records = []
    for H in HORIZON_VALUES:
        W = HORIZON_WINDOWS[H]
        records.append({
            "agent": "UrnAgent",
            "sig_n": sig_n,
            "sig_m": sig_m,
            "seed": seed,
            "horizon": H,
            "window": W,
            "final_reward": float(np.mean(rewards[0][H - W : H])),
            "final_nmi":    float(np.mean(nmi[0][H - W : H])),
        })
    return records


tasks_F = [(n, 1, s) for n in BASIN_SIG_N_VALUES for s in range(BASIN_N_SEEDS)]
print(f"Running {len(tasks_F)} Roth-Erev horizon sims "
      f"({BASIN_N_SEEDS} seeds x {len(BASIN_SIG_N_VALUES)} sig_n)...")
with tqdm_joblib(tqdm(desc="time-horizon sweep (Roth-Erev)", total=len(tasks_F))):
    records_F_nested = Parallel(n_jobs=N_JOBS)(
        delayed(run_horizon_seed)(n, m, s) for (n, m, s) in tasks_F
    )
records_F = [rec for sublist in records_F_nested for rec in sublist]
df_horizon = pd.DataFrame(records_F)
save_csv(df_horizon, "horizon_sweep_data_roth_erev.csv")
print(f"Collected {len(df_horizon)} records over "
      f"sig_n = {BASIN_SIG_N_VALUES}, horizons = {HORIZON_VALUES}")


In [ ]:
"""Option F (v3) — Roth-Erev: 1x2 grid of final reward (left) and final NMI
(right) vs initial signaling bias, with one curve per horizon (mean +/- std
shadow). Horizons colour-coded with viridis from short (dark) to long (bright)."""

import matplotlib as mpl

metrics = [("final_reward", "Final reward"),
           ("final_nmi",    "Final NMI")]

cmap = mpl.colormaps["viridis"]
horizon_colors = {H: cmap(i / max(1, len(HORIZON_VALUES) - 1))
                  for i, H in enumerate(HORIZON_VALUES)}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True, sharey=True)
for ax, (metric_col, metric_label) in zip(axes, metrics):
    for H in HORIZON_VALUES:
        df_H = df_horizon[df_horizon["horizon"] == H]
        g = df_H.groupby("sig_n")[metric_col]
        mean = g.mean()
        std = g.std()
        ax.plot(mean.index, mean.values,
                color=horizon_colors[H], lw=2, marker="o",
                label=f"{H:,} episodes")
        ax.fill_between(mean.index,
                        np.clip(mean.values - std.values, 0, 1),
                        np.clip(mean.values + std.values, 0, 1),
                        color=horizon_colors[H], alpha=0.12)
    ax.axhline(0.5, ls=":", c="grey", alpha=0.6,
               label="No-signaling baseline (0.5)" if metric_col == "final_reward" else None)
    ax.set_xscale("log")
    ax.set_xlabel("Initial signaling bias (log scale)")
    ax.set_ylabel(metric_label)
    ax.set_ylim(0, 1.05)

axes[1].legend(title="Horizon (episodes)", loc="lower right",
               fontsize=8, framealpha=0.9, ncol=2)
fig.suptitle(
    f"Roth-Erev: final reward and NMI vs initial signaling bias, by horizon  "
    f"({BASIN_N_SEEDS} trials per value; bands = mean +/- std)",
    fontsize=12,
)
plt.tight_layout()
save_and_show("horizon_sweep_roth_erev.png")


## Combined view — Option D-β and Option F side by side

Same Roth-Erev sweep across `BASIN_SIG_N_VALUES`, two perspectives in one
figure:

- **Left** — Option D-β at the deep-asymptotic horizon (10,000 episodes).
  Reward and NMI overlaid so the dissociation between them is visible at a
  glance.
- **Middle / right** — Option F across the horizon ladder
  `H ∈ {10, 50, 100, 300, 1000, 3000, 10000}`. The H=10,000 curve in each
  panel is the same data as Option D-β's mean curve; the other horizons show
  how the basin sharpens as episodes accumulate.

Read together, the panels answer: *initial signaling bias raises the basin
ceiling, but it takes thousands of episodes for the chain to actually inhabit
that basin.*

(No new compute — both panels are rebuilt from `df_basin` and `df_horizon`.)


In [ ]:
"""Combined view — Option D-beta (deep asymptotic) on the left, Option F
(multi-horizon) split across the middle and right panels. Reuses df_basin
and df_horizon from the previous sections."""

import matplotlib as mpl

cmap = mpl.colormaps["viridis"]
horizon_colors = {H: cmap(i / max(1, len(HORIZON_VALUES) - 1))
                  for i, H in enumerate(HORIZON_VALUES)}

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharex=True, sharey=True)

# --- Panel 1: D-beta (single horizon, reward + NMI overlay) -----------------
ax = axes[0]
g_r = df_basin.groupby("sig_n")["final_reward"]
mean_r, std_r = g_r.mean(), g_r.std()
ax.plot(mean_r.index, mean_r.values,
        color="firebrick", lw=2, marker="o", label="Reward (mean)")
ax.fill_between(mean_r.index,
                np.clip(mean_r.values - std_r.values, 0, 1),
                np.clip(mean_r.values + std_r.values, 0, 1),
                color="firebrick", alpha=0.18, label="Reward (mean +/- std)")

g_n = df_basin.groupby("sig_n")["final_nmi"]
mean_n, std_n = g_n.mean(), g_n.std()
ax.plot(mean_n.index, mean_n.values,
        color="darkgreen", lw=2, marker="s", label="NMI (mean)")
ax.fill_between(mean_n.index,
                np.clip(mean_n.values - std_n.values, 0, 1),
                np.clip(mean_n.values + std_n.values, 0, 1),
                color="darkgreen", alpha=0.15, label="NMI (mean +/- std)")

ax.axhline(0.5, ls=":", c="grey", alpha=0.6)
ax.set_xscale("log")
ax.set_xlabel("Initial signaling bias (log scale)")
ax.set_ylabel("Final value (last 1000 episodes)")
ax.set_title("Option D-beta -- H = 10,000")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right", fontsize=7, ncol=2, framealpha=0.9)

# --- Panels 2 & 3: Option F (multi-horizon, std bands) ----------------------
for ax, (metric_col, metric_label) in zip(
    axes[1:],
    [("final_reward", "Final reward"), ("final_nmi", "Final NMI")],
):
    for H in HORIZON_VALUES:
        df_H = df_horizon[df_horizon["horizon"] == H]
        g = df_H.groupby("sig_n")[metric_col]
        mean = g.mean()
        std = g.std()
        ax.plot(mean.index, mean.values,
                color=horizon_colors[H], lw=2, marker="o",
                label=f"{H:,} ep")
        ax.fill_between(mean.index,
                        np.clip(mean.values - std.values, 0, 1),
                        np.clip(mean.values + std.values, 0, 1),
                        color=horizon_colors[H], alpha=0.10)
    ax.axhline(0.5, ls=":", c="grey", alpha=0.6)
    ax.set_xscale("log")
    ax.set_xlabel("Initial signaling bias (log scale)")
    ax.set_title(f"Option F -- {metric_label}")

axes[2].legend(title="Horizon", loc="lower right", fontsize=7,
               framealpha=0.9, ncol=2)

fig.suptitle(
    f"Roth-Erev -- basin of attraction (D-beta) and time-horizon sweep (F) side by side  "
    f"({BASIN_N_SEEDS} trials per value)",
    fontsize=12,
)
plt.tight_layout()
save_and_show("combined_basin_and_horizon_roth_erev.png")


## Disconnect Colab runtime

On Colab, the kernel keeps the runtime billed (or against quota) until
explicitly disconnected. The cell below disconnects automatically after
all figures are rendered. On local it just prints a message and exits.


In [ ]:
"""Disconnect Colab runtime — Colab only."""

from datetime import datetime

stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

if not RUNNING_LOCALLY:
    from IPython.display import Javascript, display
    print(f"Run finished at {stamp} — disconnecting Colab runtime.")
    display(Javascript("google.colab.kernel.disconnect()"))
else:
    print(f"Run finished at {stamp}. Local mode — nothing to disconnect.")
